In [5]:
import pandas as pd

df = pd.read_csv("../data/Sample - Superstore.csv", encoding="latin1")

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
df.head()

(9994, 21)
Row ID             int64
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code        int64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity           int64
Discount         float64
Profit           float64
dtype: object
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [6]:
# 1. Column names se extra spaces hatao
df.columns = df.columns.str.strip()

# 2. Date columns ko sahi type mein badlo
df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y")

# 3. Duplicates check karo aur hatao
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

# 4. Extra time columns (dashboard ke kaam aayenge)
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.to_period("M").astype(str)

# 5. Profit margin column
df["Profit Margin %"] = (df["Profit"] / df["Sales"] * 100).round(2)

print(df.shape)
print(df.dtypes)
df.describe()

Duplicates: 0
(9994, 24)
Row ID                      int64
Order ID                      str
Order Date         datetime64[us]
Ship Date          datetime64[us]
Ship Mode                     str
Customer ID                   str
Customer Name                 str
Segment                       str
Country                       str
City                          str
State                         str
Postal Code                 int64
Region                        str
Product ID                    str
Category                      str
Sub-Category                  str
Product Name                  str
Sales                     float64
Quantity                    int64
Discount                  float64
Profit                    float64
Year                        int32
Month                         str
Profit Margin %           float64
dtype: object


,Row ID,Order Date,Ship Date,Postal Code,Sales,Quantity,Discount,Profit,Year,Profit Margin %
count,9994.000000,9994,9994,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,2016-04-30 00:07:12.259355,2016-05-03 23:06:58.571142,55190.379428,229.858001,3.789574,0.156203,28.656896,2015.722233,12.031390
min,1.000000,2014-01-03 00:00:00,2014-01-07 00:00:00,1040.000000,0.444000,1.000000,0.000000,-6599.978000,2014.000000,-275.000000
25%,2499.250000,2015-05-23 00:00:00,2015-05-27 00:00:00,23223.000000,17.280000,2.000000,0.000000,1.728750,2015.000000,7.500000
50%,4997.500000,2016-06-26 00:00:00,2016-06-29 00:00:00,56430.500000,54.490000,3.000000,0.200000,8.666500,2016.000000,27.000000
75%,7495.750000,2017-05-14 00:00:00,2017-05-18 00:00:00,90008.000000,209.940000,5.000000,0.200000,29.364000,2017.000000,36.250000
max,9994.000000,2017-12-30 00:00:00,2018-01-05 00:00:00,99301.000000,22638.480000,14.000000,0.800000,8399.976000,2017.000000,50.000000
std,2885.163629,NaN,NaN,32063.693350,623.245101,2.225110,0.206452,234.260108,1.123555,46.675436


In [7]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
margin = total_profit / total_sales * 100
orders = df["Order ID"].nunique()
customers = df["Customer ID"].nunique()
aov = total_sales / orders

print(f"Total Sales   : ${total_sales:,.0f}")
print(f"Total Profit  : ${total_profit:,.0f}")
print(f"Profit Margin : {margin:.2f}%")
print(f"Orders        : {orders}")
print(f"Customers     : {customers}")
print(f"Avg Order Val : ${aov:,.2f}")

# Category-wise aur Sub-Category-wise profit
print(df.groupby("Category")[["Sales", "Profit"]].sum().round(0))
print(df.groupby("Sub-Category")["Profit"].sum().sort_values().head(5))  # sabse bure 5

Total Sales   : $2,297,201
Total Profit  : $286,397
Profit Margin : 12.47%
Orders        : 5009
Customers     : 793
Avg Order Val : $458.61
                    Sales    Profit
Category                           
Furniture        742000.0   18451.0
Office Supplies  719047.0  122491.0
Technology       836154.0  145455.0
Sub-Category
Tables      -17725.4811
Bookcases    -3472.5560
Supplies     -1189.0995
Fasteners      949.5182
Machines      3384.7569
Name: Profit, dtype: float64


In [ ]:
import plotly.express as px

monthly = df.groupby("Month")[["Sales", "Profit"]].sum().reset_index()
fig = px.line(monthly, x="Month", y=["Sales", "Profit"], title="Monthly Sales & Profit")
fig.show()






























